# Merged Bengali compound-character model

Trains one ResNet18 on the union of three datasets, mapped into a single label space:

- `RAS-Compound-character-dataset` (119 classes, `class_mapping.csv`)
- `ekush-dataset` (122 classes, `ekush-dataset/metaData_img.csv`)
- `MatriVasha_Dataset` (120 classes, `matrivasha_mapping_draft.csv`)

Overlapping characters (e.g. `ক্ক` appears in all three) collapse into one class, giving **256 unique classes**.
Saves `best_merged_model.pth` in the same schema as `best_bengali_vit.pth`, so `predict_word.py --model best_merged_model.pth` works unchanged.

Run `caffeinate -i` in a terminal before starting the training cell — this is a multi-hour run and macOS will otherwise sleep and pause it.

In [1]:
import csv
import random
import time
import unicodedata
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

ROOT = Path.cwd()
IMG_SIZE = 224
SEED = 42
OLD_CHECKPOINT = ROOT / "best_bengali_vit.pth"
BEST_CHECKPOINT = ROOT / "best_merged_model.pth"
LAST_CHECKPOINT = ROOT / "merged_last.pth"


In [2]:
def norm(s):
    return unicodedata.normalize("NFC", s.strip())


def load_mapping(csv_path, key_col, val_col):
    mapping = {}
    with open(csv_path, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            v = row[val_col].strip()
            if v:
                mapping[row[key_col].strip()] = norm(v)
    return mapping


In [3]:
def build_manifest():
    """Scan all 3 datasets on disk, return [(path, class_idx), ...] and the sorted class list."""
    ras_map = load_mapping(ROOT / "class_mapping.csv", "class_id", "bengali_character")
    ekush_map = load_mapping(ROOT / "ekush-dataset" / "metaData_img.csv", "Folder Name", "Char Name")
    matri_map = load_mapping(ROOT / "matrivasha_mapping_draft.csv", "folder", "character")

    classes = sorted(set(ras_map.values()) | set(ekush_map.values()) | set(matri_map.values()))
    class_to_idx = {c: i for i, c in enumerate(classes)}

    samples = []
    for split in ("train", "test"):
        base = ROOT / "RAS-Compound-character-dataset" / split
        for class_id, char in ras_map.items():
            samples += [(str(p), class_to_idx[char]) for p in (base / class_id).glob("*.png")]

    ekush_base = ROOT / "ekush-dataset"
    for folder, char in ekush_map.items():
        samples += [(str(p), class_to_idx[char]) for p in (ekush_base / folder).glob("*.jpg")]

    for gender_dir in ("male", "feamale"):
        base = ROOT / "MatriVasha_Dataset" / gender_dir
        for folder, char in matri_map.items():
            samples += [(str(p), class_to_idx[char]) for p in (base / folder).glob("*.jpg")]

    return samples, classes


t0 = time.time()
samples, classes = build_manifest()
NUM_CLASSES = len(classes)
print(f"{len(samples)} images, {NUM_CLASSES} classes, scanned in {time.time()-t0:.1f}s")


681309 images, 256 classes, scanned in 0.6s


In [4]:
def stratified_split(samples, val_frac=0.1, test_frac=0.1, seed=SEED, limit_per_class=None):
    by_class = {}
    for path, label in samples:
        by_class.setdefault(label, []).append(path)

    rng = random.Random(seed)
    train, val, test = [], [], []
    for label, paths in by_class.items():
        rng.shuffle(paths)
        if limit_per_class:
            paths = paths[:limit_per_class]
        n = len(paths)
        n_test = max(1, int(n * test_frac)) if n > 2 else 0
        n_val = max(1, int(n * val_frac)) if n > 2 else 0
        test += [(p, label) for p in paths[:n_test]]
        val += [(p, label) for p in paths[n_test:n_test + n_val]]
        train += [(p, label) for p in paths[n_test + n_val:]]
    return train, val, test


# LIMIT_PER_CLASS = None uses everything (~681K images, ~10-13h on this Mac's GPU per 8 epochs).
# Set e.g. LIMIT_PER_CLASS = 300 for a much faster ~2-4h run if you want to iterate first.
LIMIT_PER_CLASS = None

train_samples, val_samples, test_samples = stratified_split(samples, limit_per_class=LIMIT_PER_CLASS)
print(f"Train {len(train_samples)} | Val {len(val_samples)} | Test {len(test_samples)}")


Train 545289 | Val 68010 | Test 68010


In [5]:
class PolarityNormalize:
    """Inverts so ink is bright on a dark background, regardless of source convention.

    ponytail: global-mean threshold, not per-blob -- good enough for these bimodal
    ink/paper images, would need real binarization for photos with uneven lighting.
    """
    def __call__(self, img):
        arr = np.array(img.convert("L"))
        border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
        if border.mean() > arr.mean():
            arr = 255 - arr
        return Image.fromarray(arr)


class CharDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path)
        return self.transform(img), label


In [6]:
train_tf = transforms.Compose([
    PolarityNormalize(),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
eval_tf = transforms.Compose([
    PolarityNormalize(),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(CharDataset(train_samples, train_tf), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=NUM_WORKERS, persistent_workers=True)
val_loader = DataLoader(CharDataset(val_samples, eval_tf), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True)
test_loader = DataLoader(CharDataset(test_samples, eval_tf), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True)


In [7]:
device = torch.device("mps") if torch.backends.mps.is_available() else (
    torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print("Device:", device)

model = models.resnet18(weights="IMAGENET1K_V1" if not OLD_CHECKPOINT.exists() else None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

if OLD_CHECKPOINT.exists():
    old = torch.load(OLD_CHECKPOINT, map_location="cpu", weights_only=False)
    old_sd, new_sd = old["state_dict"], model.state_dict()
    for k in new_sd:
        if k in old_sd and old_sd[k].shape == new_sd[k].shape:
            new_sd[k] = old_sd[k]
    model.load_state_dict(new_sd)
    print(f"Warm-started backbone from {OLD_CHECKPOINT.name}")

model = model.to(device)


Device: mps
Warm-started backbone from best_bengali_vit.pth


In [8]:
EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
RESUME = False  # set True to continue from merged_last.pth after a kernel restart

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = AdamW([
    {"params": list(model.parameters())[:-2], "lr": LR * 0.1},
    {"params": model.fc.parameters(), "lr": LR},
], weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

start_epoch = 1
best_val_acc = 0.0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

if RESUME and LAST_CHECKPOINT.exists():
    ckpt = torch.load(LAST_CHECKPOINT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    history = ckpt.get("history", history)
    print(f"Resumed from epoch {ckpt['epoch']}, best_val_acc={best_val_acc:.4f}")


In [9]:
def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train_mode:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train_mode:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total


for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train_mode=True)
    va_loss, va_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "state_dict": deepcopy(model.state_dict()),
            "class_names": classes,
            "display_names": classes,
            "img_size": IMG_SIZE,
        }, BEST_CHECKPOINT)

    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "epoch": epoch,
        "best_val_acc": best_val_acc,
        "history": history,
    }, LAST_CHECKPOINT)

    print(f"Epoch [{epoch:02d}/{EPOCHS}] "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f} | "
          f"Time: {time.time()-t0:.1f}s | Best Val: {best_val_acc:.4f}")

print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")


Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=96, pipe_handle=110)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: module '__main__' has no attribute 'CharDataset'


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
best = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(best["state_dict"])
test_loss, test_acc = run_epoch(test_loader, train_mode=False)
print(f"Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")
print(f"Saved {BEST_CHECKPOINT.name} -- use with: python predict_word.py word.png --model {BEST_CHECKPOINT.name}")
